# Who Gets Justice? Unsolved Homicides in the United States, 1980 - 2014

## A CRISP-DM analysis of the FBI Supplementary Homicide Reports (Murder Accountability Project release)

**Module**: DBMS71008 Data Analytics
**Assessment**: CA2 CRISP-DM Analytics Project
**Student**: Isabelle Lynch T00264026
**Repository**: https://github.com/isabellelynch/WhoGetsJustice-DataAnalytics-2026

**Question**: Which homicide victims are least likely to see their case cleared and how much of that gap remains once we account for how and where the killing happened.

In [ ]:
# -------------Setting up the Environment-----------------------
import sys, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
DATA = Path("../data")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

# Chart Styling
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "grey": "#8a8984", "ink": "#0b0b0b"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
                     "axes.edgecolor": "#8a8984", "axes.titleweight": "bold", "font.size": 10,
                     "axes.prop_cycle": matplotlib.cycler(color=[C["blue"], C["orange"], C["aqua"]])})
pd.set_option("display.max_columns", 40)

print("Python", platform.python_version(), "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| matplotlib", matplotlib.__version__, "| seaborn", sns.__version__)

Python 3.14.6 | pandas 3.1.0 | scikit-learn 1.9.1 | matplotlib 3.11.2 | seaborn 0.13.2


## 1. Business Understanding

### 1.1 Context and Problem
When a homicide is "cleared", police have identified and charged an offender. In the United States roughly one homicide in three is never cleared, this is a much lower rate than in other comparable countries. An uncleared homicide means no accountability for the killer, no answer for the family and a person roaming freely, who may very well kill again. Researchers and journalists have reported that clearance is not equal across victims, for example cases with Black victims and firearm killings have been reported to clear less often [Campedelli 2024; Jarvis et al. 2017]. This project asks whether that pattern appears in the national FBI data, how large the problem is and how much of it is explained by observable case characteristics (weapon, location, agency, year) rather than the victim's identity.

The problem is not to "predict whether a case will be solved" as an operational tool. A model that scores live cases by victim race would be ethically unacceptable. The model is a dianostic instrument, by fitting a classifier to historical outcomes we can measure which characteristics are associated with non-clearance after holding the others constant and test whether those associations are stable over time.

### 1.2 Stakeholders
| Stakeholder | Role | Interest |
| ----------------------------- | ------------ | ----------------------------- |
| Police agencies (via the FBI UCR programme) | Supply the data | Accurate reporting, fair comparison between agencies |
| Oversight bodies, state criminal-justice commissions, police leadership | Primary users | Where are the clearance gaps largest, to target investigative resources and audits |
| Journalists, researchers, advocacy groups (e.g. Murder Accountability Project) | Secondary users | Evidence on unequal outcomes |
| Victims' families and communities | Affected | Whether their cases receive equal effort |

### 1.3 Objectives
- **Practical Objective**: give an oversight body defensible evidence about which groups of homicide cases are least likely to be cleared, so that audits and investigative resources (cold-case units, forensic capacity) can be directed at the largest gaps.
- **Analytics Objective**: (1) describe clearance rates by victim, incident, agency and time (2) build a binary classifier for Unsolved vs Solved from information that is known when a homicide is reported and (3) use the interpretable model to estimate the adjusted gap in unsolved probablility between victim groups and the flexible model to check that the conclusions do not depend on linear assumptions.

### 1.4 Success Criteria
| Type | Criterion | Why |
| ----------- | ----------------------------- | ----------------------------- |
| Technical | Beat the majority-class baseline on recall and F1 for the Unsolved class on held-out data | The baseline predicts "Solved" for everyone and finds no unsolved cases | 
| Technical | ROC AUC >= 0.70 on the test set | Shows the features carry real signal about clearance, beyond chance (0.5) |
| Technical | Performance drops by less than 0.05 AUC when trained on the 1980-2009 and tested on 2010-2014 | The associations should be stable enough to describe recent cases |
| Practical | Adjusted victim-group gaps reported with direction and size in percentage points and compared with raw gaps | This answers "who gets justice?" in terms an oversight body can use |
| Practical/Ethical | Error rates audited by victim race and sex, model explicitly not recommended for case-level triage | Prevents the evidence from being turned into a tool that worsens the inequality |

### 1.5 Constraints and Assumptions
- **Data**: the SHR is voluntary. Some agencies report incompletely or not at all and the data ends in 2014. A case's "solved" status reflects what the agency reported to the FBI, not the later court outcomes.
- **No investigative variables**: there are no data on detective caseload, witnesses or forensic evidence, the factors that prior studies identify as most important [Wellford & Cronin 1999]. The models can therefore only explain part of clearance.
- **Privacy**: the records contain no names or addresses. The unit is a victim, aggregated to the agency and published openly.
- **Time and Compute**: single laptop, scikit-learn only, the random forest is tuned on a sample.

### 1.6 Project Plan
| CRISP-DM phase | Main activity in this project |
|----------------|-----------------------------------------------------------|
| Data Understanding | Profile 24 variables and 638,000 records. Quantify "Unknown" codes. Test which variables are only filled in because a case was solved. |
| Data Preparation | Select homicides, remove leakage columns, recode age, construct features, log every step |
| Modelling | Majority baseline - logistic regression - random forest |
| Evaluation | Recall/precision/F1/AUC, threshold trade-off, time-based validation, adjusted gaps, subgroup error audit |
| Deployment | Oversight dashboard recommendation, governance and refresh plan |
